In [1]:
import pandas as pd
import numpy as np

# load dataset 

normal_dataset_path = '../dataset/clean/normal/normal.csv'

attack_0rtt_dataset_path = '../dataset/clean/tls/attack_0rtt_dataset.csv'
attack_heartbleed_dataset_path = '../dataset/clean/tls/attack_heartbleed_dataset.csv'


attack_cert_probe_dataset_path = '../dataset/clean/probe/attack_cert_probe_dataset.csv'
attack_crypto_probe_dataset_path = '../dataset/clean/probe/attack_crypto_probe_dataset.csv'
attack_cve_probe_dataset_path = '../dataset/clean/probe/attack_cve_probe_dataset.csv'
attack_protocol_probe_dataset_path = '../dataset/clean/probe/attack_protocol_probe_dataset.csv'

attack_goldeneye_dataset_path = '../dataset/clean/dos/attack_goldeneye_dataset.csv'
attack_hulk_dataset_path = '../dataset/clean/dos/attack_hulk_dataset.csv'
attack_rst_flood_dataset_path = '../dataset/clean/dos/attack_rst_flood_dataset.csv'
attack_slowloris_dataset_path = '../dataset/clean/dos/attack_slowloris_dataset.csv'
attack_sync_flood_dataset_path = '../dataset/clean/dos/attack_sync_flood_dataset.csv'
attack_tcp_ack_dataset_path = '../dataset/clean/dos/attack_tcp_ack_dataset.csv'
attack_tors_dataset_path = '../dataset/clean/dos/attack_tors_dataset.csv'
attack_udp_dataset_path = '../dataset/clean/dos/attack_udp_dataset.csv'




normal_dataset = pd.read_csv(normal_dataset_path)

attack_0rtt_dataset = pd.read_csv(attack_0rtt_dataset_path)
attack_cert_probe_dataset = pd.read_csv(attack_cert_probe_dataset_path)
attack_crypto_probe_dataset = pd.read_csv(attack_crypto_probe_dataset_path)
attack_cve_probe_dataset = pd.read_csv(attack_cve_probe_dataset_path)
attack_goldeneye_dataset = pd.read_csv(attack_goldeneye_dataset_path)
attack_heartbleed_dataset = pd.read_csv(attack_heartbleed_dataset_path)
attack_hulk_dataset = pd.read_csv(attack_hulk_dataset_path)
attack_protocol_probe_dataset = pd.read_csv(attack_protocol_probe_dataset_path)
attack_rst_flood_dataset = pd.read_csv(attack_rst_flood_dataset_path)
attack_slowloris_dataset = pd.read_csv(attack_slowloris_dataset_path)
attack_sync_flood_dataset = pd.read_csv(attack_sync_flood_dataset_path)
attack_tcp_ack_dataset = pd.read_csv(attack_tcp_ack_dataset_path)
attack_tors_dataset = pd.read_csv(attack_tors_dataset_path)
attack_udp_dataset = pd.read_csv(attack_udp_dataset_path)


# Randomly sample data

# Normal
normal_dataset = normal_dataset.sample(n=49000, random_state=42) 

# TLS (use full)
tls_dataset = pd.concat(
    [
        attack_0rtt_dataset,
        attack_heartbleed_dataset
    ],
    axis=0,
    ignore_index=True
)

# Probe 
probe_dataset = pd.concat(
    [
        attack_cert_probe_dataset.sample(n=1000, random_state=42),
        attack_crypto_probe_dataset.sample(n=1000, random_state=42) ,
        attack_cve_probe_dataset.sample(n=2000, random_state=42) ,
        attack_protocol_probe_dataset.sample(n=1000, random_state=42) 
    ],
    axis=0,
    ignore_index=True
)

# Dos
dos_dataset = pd.concat(
    [
        attack_goldeneye_dataset.sample(n=1000, random_state=42),
        attack_hulk_dataset.sample(n=1000, random_state=42),
        attack_rst_flood_dataset,
        attack_slowloris_dataset,
        attack_sync_flood_dataset.sample(n=1000, random_state=42),
        attack_tcp_ack_dataset.sample(n=1000, random_state=42),
        attack_tors_dataset.sample(n=1000, random_state=42),
        attack_udp_dataset
    ],
    axis=0,
    ignore_index=True
)



In [2]:
normal_df = normal_dataset
attack_df = pd.concat([tls_dataset, probe_dataset, dos_dataset], ignore_index=True)

# Combine all
dataset_df = pd.concat([normal_df, attack_df], ignore_index=True)

#  Shuffle the data
dataset_df = dataset_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Result
print("Combined shape:", dataset_df.shape)
print(dataset_df['label'].value_counts())

Combined shape: (59380, 77)
label
normal    49000
ddos       5264
probe      5000
tls         116
Name: count, dtype: int64


#IDS MLP

In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, balanced_accuracy_score, f1_score

# ==== 1. Data preparation ====
X = dataset_df.drop(columns=['label']).values
y_str = dataset_df['label'].values

le = LabelEncoder()
y = le.fit_transform(y_str)
classes = le.classes_
num_classes = len(classes)
print("Classes:", classes)

# Scale features for MLP stability
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/val/test split (70/10/20)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.125, random_state=42, stratify=y_train_full
)  # 0.125 of 0.8 = 0.10

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Create DataLoaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset   = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset  = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

# ==== 2. Define the MLP model ====
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim, dropout=0.3):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dims[1], hidden_dims[2]),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dims[2], output_dim)
        )

    def forward(self, x):
        return self.model(x)

input_dim = X_train.shape[1]       # 77 features
hidden_dims = [128, 64, 32]        # 3 hidden layers
output_dim = num_classes           # 4 classes

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLP(input_dim, hidden_dims, output_dim, dropout=0.3).to(device)

# ==== 3. Loss function & optimizer ====
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)  # weight_decay = L2 reg

# ==== 4. Training loop ====
epochs = 50
best_val_acc = 0.0

for epoch in range(1, epochs + 1):
    # Train
    model.train()
    running_loss = 0.0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        outputs = model(Xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Validation
    # Validation
    model.eval()
    val_loss = 0.0
    val_correct, val_total = 0, 0
    val_true, val_pred = [], []

    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            outputs = model(Xb)
            loss = criterion(outputs, yb)
            val_loss += loss.item()

            preds = outputs.argmax(dim=1)
            val_correct += (preds == yb).sum().item()
            val_total += yb.size(0)

            val_true.extend(yb.cpu().numpy())
            val_pred.extend(preds.cpu().numpy())

    val_acc = val_correct / val_total
    val_bal_acc = balanced_accuracy_score(val_true, val_pred)

    # keep your checkpointing logic (still using val_acc, or switch to val_bal_acc if you prefer)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_mlp.pth")

    print(f"Epoch [{epoch}/{epochs}] "
        f"Train Loss: {running_loss/len(train_loader):.4f}  "
        f"Val Loss: {val_loss/len(val_loader):.4f}  "
        f"Val Acc: {val_acc:.4f}  "
        f"Val Balanced Acc: {val_bal_acc:.4f}")

# ==== 5. Testing ====
model.load_state_dict(torch.load("best_mlp.pth"))
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for Xb, yb in test_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        outputs = model(Xb)
        preds = outputs.argmax(dim=1)
        y_true.extend(yb.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print("TEST accuracy:", accuracy_score(y_true, y_pred))
print("TEST balanced_accuracy:", balanced_accuracy_score(y_true, y_pred))
print("TEST macro F1:", f1_score(y_true, y_pred, average="macro"))
print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=classes))

Classes: ['ddos' 'normal' 'probe' 'tls']
Epoch [1/50] Train Loss: 0.1062  Val Loss: 0.0304  Val Acc: 0.9941  Val Balanced Acc: 0.7426
Epoch [2/50] Train Loss: 0.0353  Val Loss: 0.0261  Val Acc: 0.9949  Val Balanced Acc: 0.7425
Epoch [3/50] Train Loss: 0.0297  Val Loss: 0.0255  Val Acc: 0.9946  Val Balanced Acc: 0.7428
Epoch [4/50] Train Loss: 0.0276  Val Loss: 0.0203  Val Acc: 0.9951  Val Balanced Acc: 0.7425
Epoch [5/50] Train Loss: 0.0251  Val Loss: 0.0190  Val Acc: 0.9951  Val Balanced Acc: 0.7633
Epoch [6/50] Train Loss: 0.0216  Val Loss: 0.0171  Val Acc: 0.9953  Val Balanced Acc: 0.7634
Epoch [7/50] Train Loss: 0.0212  Val Loss: 0.0162  Val Acc: 0.9958  Val Balanced Acc: 0.8259
Epoch [8/50] Train Loss: 0.0196  Val Loss: 0.0181  Val Acc: 0.9958  Val Balanced Acc: 0.8259
Epoch [9/50] Train Loss: 0.0195  Val Loss: 0.0142  Val Acc: 0.9958  Val Balanced Acc: 0.8259
Epoch [10/50] Train Loss: 0.0190  Val Loss: 0.0172  Val Acc: 0.9960  Val Balanced Acc: 0.8259
Epoch [11/50] Train Loss: 0.

/tmp/ipykernel_3276613/3257840334.py:136: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_mlp.pth"))
